In [0]:
%sql
-- CELL 1: Create State Lookup Table
CREATE OR REPLACE TABLE cms.silver.ref_state
COMMENT 'Prescriber state codes: names and area type. Covers every code CMS documents, not only those observed.'
AS
SELECT * FROM VALUES
  ('AL','Alabama','State'),            ('AK','Alaska','State'),
  ('AZ','Arizona','State'),            ('AR','Arkansas','State'),
  ('CA','California','State'),         ('CO','Colorado','State'),
  ('CT','Connecticut','State'),        ('DE','Delaware','State'),
  ('FL','Florida','State'),            ('GA','Georgia','State'),
  ('HI','Hawaii','State'),             ('ID','Idaho','State'),
  ('IL','Illinois','State'),           ('IN','Indiana','State'),
  ('IA','Iowa','State'),               ('KS','Kansas','State'),
  ('KY','Kentucky','State'),           ('LA','Louisiana','State'),
  ('ME','Maine','State'),              ('MD','Maryland','State'),
  ('MA','Massachusetts','State'),      ('MI','Michigan','State'),
  ('MN','Minnesota','State'),          ('MS','Mississippi','State'),
  ('MO','Missouri','State'),           ('MT','Montana','State'),
  ('NE','Nebraska','State'),           ('NV','Nevada','State'),
  ('NH','New Hampshire','State'),      ('NJ','New Jersey','State'),
  ('NM','New Mexico','State'),         ('NY','New York','State'),
  ('NC','North Carolina','State'),     ('ND','North Dakota','State'),
  ('OH','Ohio','State'),               ('OK','Oklahoma','State'),
  ('OR','Oregon','State'),             ('PA','Pennsylvania','State'),
  ('RI','Rhode Island','State'),       ('SC','South Carolina','State'),
  ('SD','South Dakota','State'),       ('TN','Tennessee','State'),
  ('TX','Texas','State'),              ('UT','Utah','State'),
  ('VT','Vermont','State'),            ('VA','Virginia','State'),
  ('WA','Washington','State'),         ('WV','West Virginia','State'),
  ('WI','Wisconsin','State'),          ('WY','Wyoming','State'),
  ('DC','District of Columbia','District'),
  ('PR','Puerto Rico','Territory'),    ('VI','U.S. Virgin Islands','Territory'),
  ('GU','Guam','Territory'),           ('MP','Northern Mariana Islands','Territory'),
  ('AS','American Samoa','Territory'),
  ('AA','Armed Forces Americas','Military'),
  ('AE','Armed Forces Europe','Military'),
  ('AP','Armed Forces Pacific','Military'),
  ('FM','Federated States of Micronesia','Freely Associated State'),
  ('MH','Marshall Islands','Freely Associated State'),
  ('PW','Palau','Freely Associated State'),
  ('ZZ','Foreign Country','Foreign'),
  ('XX','Unknown','Unknown')
AS t(state_code, state_name, area_type);

-- In Bronze, we were missing one of the territories. This table's job is to say what each code means. 

In [0]:
## CELL 2: Checks for no duplicate codes and that every code in the data has a match.

ref = spark.table("cms.silver.ref_state")
assert ref.count() == ref.select("state_code").distinct().count(), "Duplicate codes in ref_state"

unmatched = spark.sql("""
SELECT s.state_code,
       COUNT(*)                        AS n_rows,
       COUNT(DISTINCT s.prescriber_npi) AS n_prescribers
FROM cms.silver.partd_prescriber_drug AS s
LEFT ANTI JOIN cms.silver.ref_state AS r
  ON s.state_code = r.state_code
GROUP BY s.state_code
""").collect()

assert not unmatched, "State codes missing from ref_state:\n" + "\n".join(
    f"  {r.state_code!r}: {r.n_rows:,} rows, {r.n_prescribers:,} prescribers" for r in unmatched
)
print("Every state code in silver maps to ref_state.")

## This code is designed to fail. We already know one of these is missing. FM is the Federated States of Micronesia. It’s a “freely associated state”: an independent country with a special agreement with the U.S. under which U.S. postal codes and some federal programs apply. That’s how one prescriber there ended up in Medicare Part D. The Marshall Islands (MH) and Palau (PW) have the same arrangement and their own postal codes, so they go in too, for the same reason American Samoa did: the lookup describes every possible code, not just the ones that show up.

In [0]:
## CELL 3: Specialty Lookup

dental_specialties = [
    "Dentist",
    "Oral Surgery (Dentist only)",
    "Oral & Maxillofacial Surgery",
    "Dental Hygienist",
    "Oral Medicinist",
    "Oral and Maxillofacial Pathology",
    "Denturist",
    "Dental Assistant",
    "Advanced Practice Dental Therapist",
    "Oral Medicine",
    "Dental Laboratory Technician",
    "Dental Therapist",
]

# Labels the dental pattern matches that a human reviewed and ruled non-dental
reviewed_non_dental = [
    "Student in an Organized Health Care Education/Training Program",
    "Community/Behavioral Health",
    "Independent Medical Examiner",
]

(spark.createDataFrame([(s,) for s in dental_specialties], "specialty STRING")
      .createOrReplaceTempView("dental_specialties"))

## Cell 5 in Bronze had false positives, like LIKE '%dent%' matched “Student” and “Independent,” and '%oral%' matched “Behavioral.” So, we create groups by hand, pointing out the false-positives and then the rest going into specialty. 

## createOrReplaceTempView registers the Python list as a temporary table named dental_specialties. A %sql cell can then query it like any other table. It isn’t saved to the catalog, and it disappears when the notebook session ends.


In [0]:
%sql
-- CELL 4: Build Lookup Specialty Table, using the dental list to assign the groups. 
CREATE OR REPLACE TABLE cms.silver.ref_specialty
COMMENT 'Every observed prescriber specialty with its group. Dental membership is an explicit list, not a pattern.'
AS
SELECT
  s.specialty,
  CASE WHEN d.specialty IS NOT NULL THEN 'Dental' ELSE 'Other' END AS specialty_group
FROM (SELECT DISTINCT specialty FROM cms.silver.partd_prescriber_drug) AS s
LEFT JOIN dental_specialties AS d
  ON s.specialty = d.specialty;

 -- This cell depends on the last one.  

In [0]:
## CELL 3: The Checks

ref = spark.table("cms.silver.ref_specialty")

# 1. One row per label
assert ref.count() == ref.select("specialty").distinct().count(), "Duplicate specialty labels"

# 2. Every typed dental label actually exists in the data (catches typos)
found = {r.specialty for r in ref.filter("specialty_group = 'Dental'").collect()}
missing = set(dental_specialties) - found
assert not missing, f"Dental labels not found in data (typo or renamed?): {missing}"

# 3. No rows without a specialty
null_n = spark.table("cms.silver.partd_prescriber_drug").filter("specialty IS NULL").count()
assert null_n == 0, f"{null_n} rows have no specialty"

# 4. Alarm: dental-sounding labels nobody has classified
suspects = [
    r.specialty
    for r in ref.filter("specialty_group = 'Other' AND specialty RLIKE '(?i)dent|oral|odont|maxillofacial'").collect()
    if r.specialty not in reviewed_non_dental
]
assert not suspects, f"Unreviewed dental-looking labels: {suspects}"

print(f"ref_specialty: {ref.count()} labels, {len(found)} dental. All checks passed.")

## It's going to catch the null values, and a NULL key matches nothing in a join. ref_specialty already has a NULL row. SELECT DISTINCT includes NULL as one of its values, so Cell 4 quietly created a lookup entry with no label.